In [0]:
USE CATALOG smart_odoo;


CREATE TABLE IF NOT EXISTS gold.fact_inventory
(
    -- Keys
    stock_quant_id          INT,
    product_id              INT,
    product_name            STRING,
    categ_id                INT,
    categ_name              STRING,
    category                STRING,
    product_type            STRING,       -- 'Laptop' | 'Accessory'

    warehouse_id            INT,
    warehouse_name          STRING,
    location_id             INT,
    location_name           STRING,
    location_type           STRING,
    lot_id                  INT,
    company_id              INT,
    company_name            STRING,

    -- Inventory Metrics
    quantity_on_hand        DOUBLE,
    reserved_quantity       DOUBLE,
    available_quantity      DOUBLE,
    available_pct           DOUBLE,       -- available / on_hand  (0-1)
    inventory_diff_quantity DOUBLE,
    inventory_quantity_set  BOOLEAN,

    -- Valuation  (quantity_on_hand × unit_cost)
    unit_cost               DOUBLE,
    stock_value             DOUBLE,

    -- Reorder Rules
    product_min_qty         DOUBLE,
    product_max_qty         DOUBLE,
    qty_to_reorder          DOUBLE,       -- MAX(0, max_qty - available_qty) when low

    -- Inventory Status
    inventory_status        STRING,       -- 'Out of Stock' | 'Low Stock' | 'In Stock'
    is_low_stock            BOOLEAN,
    is_out_of_stock         BOOLEAN,
    inventory_age_days      DOUBLE,
    inventory_movement_status            STRING,

    -- Dates
    inventory_date          TIMESTAMP,
    in_date                 TIMESTAMP,
    accounting_date         TIMESTAMP,
    created_at              TIMESTAMP,
    updated_at              TIMESTAMP,
    dwh_loaded_at           TIMESTAMP
)
USING DELTA;

-- ── MERGE ────────────────────────────────────────────────────────────────────

MERGE INTO gold.fact_inventory AS target

USING
(
    SELECT
        -- Keys
        CAST(sq.stock_quant_id  AS INT)    AS stock_quant_id,
        CAST(sq.product_id      AS INT)    AS product_id,
        sq.product_name,

        CAST(pt.categ_id        AS INT)    AS categ_id,
        pt.categ_name,
        TRIM(SPLIT(pt.categ_name, '/')[SIZE(SPLIT(categ_name, '/')) - 1]) AS category,

        pt.type                            AS product_type,

        CAST(sw.warehouse_id    AS INT)    AS warehouse_id,
        sw.warehouse_name,
        CAST(sl.stock_location_id AS INT)  AS location_id,
        sl.stock_location_name             AS location_name,
        sl.usage                           AS location_type,

        CAST(sq.lot_id          AS INT)    AS lot_id,
        CAST(sq.company_id      AS INT)    AS company_id,
        sq.company_name,

        -- Inventory Metrics
        CAST(sq.quantity           AS DOUBLE) AS quantity_on_hand,
        CAST(sq.reserved_quantity  AS DOUBLE) AS reserved_quantity,

        CAST(sq.quantity - sq.reserved_quantity AS DOUBLE)
                                           AS available_quantity,

        CAST(
            (sq.quantity - sq.reserved_quantity)
            / NULLIF(sq.quantity, 0)
        AS DOUBLE)                         AS available_pct,

        sq.inventory_diff_quantity,
        sq.inventory_quantity_set,         -- FIX 2: Boolean, not diff value

        -- Cost & Valuation
        -- FIX 3: use receipt-only cost (is_in = TRUE) so we never mix
        --        in list_price from delivery moves.
        --        Falls back to product standard_price when no receipts exist.
        CAST(
            COALESCE(receipt_cost.avg_unit_cost, pp.standard_price, 0.0)
        AS DOUBLE)                         AS unit_cost,

        CAST(
            sq.quantity
            * COALESCE(receipt_cost.avg_unit_cost, pp.standard_price, 0.0)
        AS DOUBLE)                         AS stock_value,   -- FIX 4: on_hand × cost

        -- Reorder Rules
        op.product_min_qty,
        op.product_max_qty,

        -- ADD 6: how much to order to reach max_qty
        CAST(
            CASE
                WHEN (sq.quantity - sq.reserved_quantity)
                        < COALESCE(op.product_min_qty, 0)
                THEN GREATEST(0.0,
                        COALESCE(op.product_max_qty, 0)
                        - (sq.quantity - sq.reserved_quantity))
                ELSE 0.0
            END
        AS DOUBLE)                         AS qty_to_reorder,

        -- Inventory Status
        CASE
            WHEN sq.quantity <= 0
                THEN 'Out of Stock'
            WHEN (sq.quantity - sq.reserved_quantity)
                    < COALESCE(op.product_min_qty, 20)
                THEN 'Low Stock'
            ELSE 'In Stock'
        END                                AS inventory_status,

        CASE
            WHEN (sq.quantity - sq.reserved_quantity)
                    < COALESCE(op.product_min_qty, 20)
            THEN TRUE ELSE FALSE
        END                                AS is_low_stock,

        CASE
            WHEN sq.quantity <= 0
            THEN TRUE ELSE FALSE
        END  
                                      AS is_out_of_stock,
        DATEDIFF(
        CURRENT_DATE(),
        CAST(sq.in_date AS DATE)
        ) AS inventory_age_days,

        CASE
            WHEN DATEDIFF(
                CURRENT_DATE(),
                CAST(sq.in_date AS DATE)
            ) <= 30
            THEN 'Fast Moving'

            WHEN DATEDIFF(
                CURRENT_DATE(),
                CAST(sq.in_date AS DATE)
            ) <= 90
            THEN 'Normal Moving'

            WHEN DATEDIFF(
                CURRENT_DATE(),
                CAST(sq.in_date AS DATE)
            ) <= 180
            THEN 'Slow Moving'

            ELSE 'Dead Stock'

        END AS inventory_movement_status,


        -- Dates
        sq.inventory_date,
        sq.in_date,
        sq.accounting_date,
        sq.created_at,
        sq.updated_at,
        CURRENT_TIMESTAMP()                AS dwh_loaded_at

    FROM silver.stock_quant sq

    -- Location details
    LEFT JOIN silver.stock_location sl
        ON sl.stock_location_id = sq.location_id

    -- Warehouse details
    LEFT JOIN silver.stock_warehouse sw
        ON sw.warehouse_id = sl.warehouse_id

    -- ADD 5: Product category & type
    LEFT JOIN silver.product_template pt
        ON pt.product_tmpl_id = sq.product_id

    -- Product cost (standard_price fallback)
    LEFT JOIN silver.product_product pp
        ON pp.product_id = sq.product_id

    -- Reorder rules
    LEFT JOIN silver.stock_warehouse_orderpoint op
        ON  op.product_id  = sq.product_id
        AND op.location_id = sq.location_id
        AND op.active      = TRUE

    -- FIX 3: receipt-only average unit cost
    LEFT JOIN
    (
        SELECT
            product_id,
            AVG(price_unit) AS avg_unit_cost
        FROM silver.stock_move
        WHERE state    = 'done'
          AND is_in    = TRUE          -- receipts only, never delivery price
          AND price_unit > 0
        GROUP BY product_id
    ) receipt_cost
        ON receipt_cost.product_id = sq.product_id

    -- Internal stock locations only
    WHERE sl.usage = 'internal'

) AS source

ON target.stock_quant_id = source.stock_quant_id

WHEN MATCHED
 AND target.updated_at < source.updated_at
THEN UPDATE SET
    target.product_id              = source.product_id,
    target.product_name            = source.product_name,
    target.categ_id                = source.categ_id,
    target.categ_name              = source.categ_name,
    target.category                = source.category,
    target.product_type            = source.product_type,
    target.warehouse_id            = source.warehouse_id,
    target.warehouse_name          = source.warehouse_name,
    target.location_id             = source.location_id,
    target.location_name           = source.location_name,
    target.location_type           = source.location_type,
    target.lot_id                  = source.lot_id,
    target.company_id              = source.company_id,
    target.company_name            = source.company_name,
    target.quantity_on_hand        = source.quantity_on_hand,
    target.reserved_quantity       = source.reserved_quantity,
    target.available_quantity      = source.available_quantity,
    target.available_pct           = source.available_pct,
    target.inventory_diff_quantity = source.inventory_diff_quantity,
    target.inventory_quantity_set  = source.inventory_quantity_set,  -- FIX 2
    target.unit_cost               = source.unit_cost,
    target.stock_value             = source.stock_value,
    target.product_min_qty         = source.product_min_qty,
    target.product_max_qty         = source.product_max_qty,
    target.qty_to_reorder          = source.qty_to_reorder,
    target.inventory_status        = source.inventory_status,
    target.is_low_stock            = source.is_low_stock,
    target.is_out_of_stock         = source.is_out_of_stock,
    target.inventory_age_days      = source.inventory_age_days,
    target.inventory_movement_status      = source.inventory_movement_status,
    target.inventory_date          = source.inventory_date,
    target.in_date                 = source.in_date,
    target.accounting_date         = source.accounting_date,
    target.created_at              = source.created_at,
    target.updated_at              = source.updated_at,
    target.dwh_loaded_at           = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT *;


In [0]:
USE CATALOG smart_odoo;

CREATE TABLE IF NOT EXISTS gold.fact_inventory_movement
(
    stock_move_line_id  INT,

    -- Product
    product_id          INT,
    product_name        STRING,

    -- Keys
    warehouse_id        INT,
    lot_id              INT,
    picking_id          INT,

    -- Location
    from_location_id    INT,              -- FIX 8: INT not STRING
    to_location_id      INT,              -- FIX 8: INT not STRING
    from_location_name  STRING,
    to_location_name    STRING,

    -- Partner (vendor for receipts, customer for deliveries)
    partner_id          INT,              -- ADD 12
    partner_name        STRING,           -- ADD 12

    -- Quantities  (FIX 9: two separate columns)
    ordered_qty         DOUBLE,           -- product_uom_qty  (demand)
    moved_qty           DOUBLE,           -- quantity         (actual transfer, 0 if not done)

    unit_price          DOUBLE,
    move_value          DOUBLE,           -- ordered_qty × unit_price

    -- Descriptive
    lot_name            STRING,
    picking_name        STRING,
    origin              STRING,           -- ADD 13: SO-XXXX / PO-XXXX
    movement_type       STRING,           -- ADD 11: Receipt / Delivery / Internal Transfer / Return
    move_state          STRING,
    picking_state       STRING,

    -- Direction flags
    is_in               BOOLEAN,
    is_out              BOOLEAN,
    is_internal         BOOLEAN,
    is_return           BOOLEAN,          -- ADD 11

    -- Dates
    movement_date       DATE,
    created_at          TIMESTAMP,
    updated_at          TIMESTAMP,
    dwh_loaded_at       TIMESTAMP
)
USING DELTA;

-- ── MERGE ────────────────────────────────────────────────────────────────────

MERGE INTO gold.fact_inventory_movement AS target

USING
(
    SELECT
        CAST(sml.stock_move_line_id AS INT)   AS stock_move_line_id,

        -- Product  (ADD 14)
        CAST(sm.product_id AS INT)            AS product_id,
        sm.product_name,

        -- Keys
        CAST(sm.warehouse_id AS INT)          AS warehouse_id,
        CAST(sml.lot_id      AS INT)          AS lot_id,
        CAST(sml.picking_id  AS INT)          AS picking_id,    -- FIX 10: was un-cast STRING

        -- Location  (FIX 8: INT)
        CAST(sm.location_id      AS INT)      AS from_location_id,
        CAST(sm.location_dest_id AS INT)      AS to_location_id,
        src_loc.stock_location_name           AS from_location_name,
        dst_loc.stock_location_name           AS to_location_name,

        -- Partner  (ADD 12)
        CAST(sp.partner_id AS INT)            AS partner_id,
        sp.partner_name,

        -- Quantities  (FIX 9)
        CAST(sml.quantity_product_uom AS DOUBLE) AS ordered_qty,  -- demand
        CAST(sml.quantity             AS DOUBLE) AS moved_qty,    -- actual (0 if not done)

        CAST(sm.price_unit AS DOUBLE)         AS unit_price,
        CAST(sml.quantity_product_uom AS DOUBLE)
            * CAST(sm.price_unit AS DOUBLE)   AS move_value,      -- based on ordered

        -- Descriptive
        sml.lot_name,
        sp.stock_picking_name                 AS picking_name,
        sp.origin,                                                 -- ADD 13

        -- ADD 11: readable movement type
        CASE sp.picking_type_id
            WHEN 1 THEN 'Receipt'
            WHEN 2 THEN 'Delivery'
            WHEN 3 THEN 'Internal Transfer'
            WHEN 4 THEN 'Return'
            ELSE        'Other'
        END                                   AS movement_type,

        sm.state                              AS move_state,
        sp.state                              AS picking_state,

        -- Direction flags
        CAST(sm.is_in  AS BOOLEAN)            AS is_in,
        CAST(sm.is_out AS BOOLEAN)            AS is_out,
        (NOT CAST(sm.is_in AS BOOLEAN)
         AND NOT CAST(sm.is_out AS BOOLEAN))  AS is_internal,
        (sp.picking_type_id = 4)              AS is_return,       -- ADD 11

        -- Dates
        CAST(sml.date AS DATE)                AS movement_date,
        sml.created_at,
        sml.updated_at,
        CURRENT_TIMESTAMP()                   AS dwh_loaded_at

    FROM silver.stock_move_line sml

    LEFT JOIN silver.stock_move sm
        ON sm.stock_move_id = sml.move_id

    LEFT JOIN silver.stock_picking sp
        ON sp.stock_picking_id = CAST(sml.picking_id AS INT)     -- FIX 10

    LEFT JOIN silver.stock_location src_loc
        ON src_loc.stock_location_id = sm.location_id

    LEFT JOIN silver.stock_location dst_loc
        ON dst_loc.stock_location_id = sm.location_dest_id

    WHERE sml.updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.fact_inventory_movement),
        CAST('1900-01-01' AS TIMESTAMP)
    )

) AS source

ON target.stock_move_line_id = source.stock_move_line_id

WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.product_id        = source.product_id,
    target.product_name      = source.product_name,
    target.warehouse_id      = source.warehouse_id,
    target.lot_id            = source.lot_id,
    target.picking_id        = source.picking_id,
    target.from_location_id  = source.from_location_id,
    target.to_location_id    = source.to_location_id,
    target.from_location_name= source.from_location_name,
    target.to_location_name  = source.to_location_name,
    target.partner_id        = source.partner_id,
    target.partner_name      = source.partner_name,
    target.ordered_qty       = source.ordered_qty,
    target.moved_qty         = source.moved_qty,
    target.unit_price        = source.unit_price,
    target.move_value        = source.move_value,
    target.lot_name          = source.lot_name,
    target.picking_name      = source.picking_name,
    target.origin            = source.origin,
    target.movement_type     = source.movement_type,
    target.move_state        = source.move_state,
    target.picking_state     = source.picking_state,
    target.is_in             = source.is_in,
    target.is_out            = source.is_out,
    target.is_internal       = source.is_internal,
    target.is_return         = source.is_return,
    target.movement_date     = source.movement_date,
    target.created_at        = source.created_at,
    target.updated_at        = source.updated_at,
    target.dwh_loaded_at     = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT *;


In [0]:
USE CATALOG smart_odoo;

-- ═══════════════════════════════════════════════════════════════════════
-- DIM: gold.dim_inventory_location
--
-- FIXES vs previous version
-- ──────────────────────────
-- FIX 15: created_at / updated_at kept as TIMESTAMP (were DATE — breaks
--         incremental filter precision)
-- ADD 16: barcode column
-- ADD 17: replenish_location flag
-- ═══════════════════════════════════════════════════════════════════════

CREATE TABLE IF NOT EXISTS gold.dim_inventory_location
(
    location_id           INT,
    warehouse_id          INT,

    location_name         STRING,
    warehouse_name        STRING,
    location_path         STRING,
    location_type         STRING,

    barcode               STRING,           -- ADD 16
    replenish_location    BOOLEAN,          -- ADD 17

    is_active             BOOLEAN,

    created_at            TIMESTAMP,        -- FIX 15: TIMESTAMP not DATE
    updated_at            TIMESTAMP,        -- FIX 15: TIMESTAMP not DATE
    dwh_loaded_at         TIMESTAMP
)
USING DELTA;

-- ── MERGE ────────────────────────────────────────────────────────────────────

MERGE INTO gold.dim_inventory_location AS target

USING
(
    SELECT
        CAST(sl.stock_location_id AS INT) AS location_id,
        CAST(sw.warehouse_id      AS INT) AS warehouse_id,

        sl.stock_location_name            AS location_name,
        sw.warehouse_name,
        sl.complete_name                  AS location_path,
        sl.usage                          AS location_type,

        sl.barcode,                                         -- ADD 16
        sl.replenish_location,                              -- ADD 17

        CAST(sl.active AS BOOLEAN)        AS is_active,

        sl.created_at,                                      -- FIX 15: full TIMESTAMP
        sl.updated_at,
        CURRENT_TIMESTAMP()               AS dwh_loaded_at

    FROM silver.stock_location sl
    LEFT JOIN silver.stock_warehouse sw
        ON sw.warehouse_id = sl.warehouse_id

    WHERE sl.updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.dim_inventory_location),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.location_id = source.location_id

WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.warehouse_id        = source.warehouse_id,
    target.location_name       = source.location_name,
    target.warehouse_name      = source.warehouse_name,
    target.location_path       = source.location_path,
    target.location_type       = source.location_type,
    target.barcode             = source.barcode,
    target.replenish_location  = source.replenish_location,
    target.is_active           = source.is_active,
    target.updated_at          = source.updated_at,
    target.dwh_loaded_at       = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT *;


In [0]:
USE CATALOG smart_odoo;

CREATE TABLE IF NOT EXISTS gold.dim_warehouse
(
    warehouse_id      INT,
    company_id        INT,
    company_name      STRING,

    warehouse_name    STRING,
    warehouse_code    STRING,
    region            STRING,             -- ADD 20: city/region label

    partner_id        INT,               -- ADD 19
    partner_name      STRING,            -- ADD 19

    reception_steps   STRING,
    delivery_steps    STRING,

    is_active         BOOLEAN,

    created_at        TIMESTAMP,          -- FIX 18: TIMESTAMP not DATE
    updated_at        TIMESTAMP,          -- FIX 18: TIMESTAMP not DATE
    dwh_loaded_at     TIMESTAMP
)
USING DELTA;

-- ── MERGE ────────────────────────────────────────────────────────────────────

MERGE INTO gold.dim_warehouse AS target

USING
(
    SELECT
        CAST(w.warehouse_id AS INT) AS warehouse_id,
        CAST(w.company_id   AS INT) AS company_id,
        w.company_name,

        w.warehouse_name,
        w.code AS warehouse_code,

        -- ADD 20: human-readable region from code prefix
        CASE
            WHEN w.code LIKE 'CAI%' THEN 'Cairo'
            WHEN w.code LIKE 'ALX%' THEN 'Alexandria'
            WHEN w.code LIKE 'GZ%'  THEN 'Giza'
            WHEN w.code LIKE 'DLT%' THEN 'Delta'
            WHEN w.code LIKE 'MNS%' THEN 'Mansoura'
            WHEN w.code LIKE 'TNT%' THEN 'Tanta'
            WHEN w.code LIKE 'ASN%' THEN 'Aswan'
            ELSE 'Other'
        END AS region,

        -- ADD 19: warehouse contact
        CAST(w.partner_id AS INT)   AS partner_id,
        w.partner_name,

        w.reception_steps,
        w.delivery_steps,

        CAST(w.active AS BOOLEAN)   AS is_active,

        w.created_at,                                       -- FIX 18
        w.updated_at,
        CURRENT_TIMESTAMP()         AS dwh_loaded_at

    FROM silver.stock_warehouse w

    WHERE w.updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.dim_warehouse),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.warehouse_id = source.warehouse_id

WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.company_id      = source.company_id,
    target.company_name    = source.company_name,
    target.warehouse_name  = source.warehouse_name,
    target.warehouse_code  = source.warehouse_code,
    target.region          = source.region,
    target.partner_id      = source.partner_id,
    target.partner_name    = source.partner_name,
    target.reception_steps = source.reception_steps,
    target.delivery_steps  = source.delivery_steps,
    target.is_active       = source.is_active,
    target.updated_at      = source.updated_at,
    target.dwh_loaded_at   = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT *;
